# 03 — Composite Validation (multi-cycle, DCC process **V1**)
**SEA-FORWARD** · OPERA Capacity Development · OceanPrediction-A toolkit

`02_validation.ipynb` validates ONE forecast cycle, day by calendar date.
This notebook instead **merges several cycles** into one composite
analysis, indexed by **forecast lead time** rather than calendar date:

    cycle 20260701's days -> sp1, sp2, fcst1, fcst2, fcst3, ...
    cycle 20260706's days -> sp1, sp2, fcst1, fcst2, fcst3, ...
    cycle 20260711's days -> sp1, sp2, fcst1, fcst2, fcst3, fcst4
        |
        v  merged/pooled BY LABEL, not by calendar date
    sp1, sp2 : model spin-up (excluded from pass/fail, same convention as
               02_validation.ipynb Section 6/SPINUP_DAYS)
    fcst1, fcst2, fcst3, ... : genuine forecast lead days, POOLED across
               every cycle that reaches that lead

so "how skillful is the forecast on its 3rd day" can be assessed across
every cycle at once, instead of one cycle's absolute dates. Copernicus
Marine and satellite reference data are downloaded/checked per cycle
(same logic as 02_validation.ipynb Section 1c) and merged the same way --
by lead label, not calendar date.

Everything here is a thin wrapper around `sftools.validation_composite`
(`vc`), which itself is a thin per-lead ACCUMULATOR around the same
`sftools.validation` (`val`) building blocks 02_validation.ipynb uses --
so a composite run and a single-cycle run can never silently disagree on
methodology.


In [1]:
# ----------------------------------------------------------------------
# Setup -- run from the notebooks/ folder so sftools imports (see sftools/README.md)
# ----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, "..")   # repo root, so `import sftools...` resolves

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import glob
import uuid

import sftools.postprocess as pp
import sftools.validation as val               # single-cycle building blocks (maps,
                                               # scatter, GODAE metrics, satellite,
                                               # HTML summary, ...)
import sftools.validation_composite as vc      # composite (multi-cycle, lead-time) layer
import sftools.plotting as pl
from   sftools.download import cmems

import _paths


## 0. Cycles to composite

Every forecast run lives in a cycle directory named `YYYYMMDD` (ex: ```20260729```), sibling
to every other cycle under `<MAIN_DIR>/<CONFIG>/` (see
`notebooks/_paths.py`). Pick **at least 2** cycles below to merge into one
composite -- they don't need to be the same length (a 5-day and a 7-day
forecast cycle can be composited together; the 7-day one just contributes
to `fcst6`/`fcst7` on its own, see Section 1c).


In [2]:
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
MAIN_DIR = os.path.expanduser(
    os.environ.get("SEAFORWARD_MAIN_DIR", "~/seaforward/forecast/model-runs"))

AVAILABLE_CYCLES = _paths.list_cycles(MAIN_DIR, CONFIG)
print(f"Forecast cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")


Forecast cycles found under /home/lell/seaforward/forecast/model-runs/Canary_12: ['20260711', '20260723', '20260729']


In [3]:
# >>> SET THIS to the cycles you want to composite, e.g. at least 2 <
# Leave the env var unset AND the list below empty for EVERY available cycle
# under MAIN_DIR/CONFIG to be selected (AVAILABLE_CYCLES above).
CYCLES = os.environ.get("SEAFORWARD_CYCLES", "").split(",") if os.environ.get("SEAFORWARD_CYCLES") \
        else []   # <<<< write here at least two cycles you wish to validate
if not CYCLES:
    CYCLES = list(AVAILABLE_CYCLES)

if len(CYCLES) < 2:
    raise ValueError(f"CYCLES must have at least 2 entries for a composite validation, got {CYCLES!r}")

YORIG = 2000   # if not working, set to >>> None <<< since real CROCO output carries proper CF time units

# >>> Depth level for the grid comparisons in Section 2 (SST/currents/SSS maps) <
# None -> surface. SSH has no depth dimension, always surface regardless of this.
DEPTH_M = None

# The first SPINUP_DAYS day(s) of EVERY cycle carry CROCO's model start-up
# transient -- labelled 'sp1','sp2',... (kept separate from 'fcst1','fcst2',...)
# so they're visible in the boxplots (Section 5b/5c) but excluded from the
# pass/fail summary (Section 5), exactly like 02_validation.ipynb Section 6.
SPINUP_DAYS = 2

# coordinates for point timeseries and vertical profiles
POINT_LON, POINT_LAT = -17.0, 18.0   # <<<< set to your point of interest


```COMPOSITE_DIR``` is named with a short uuid rather than the cycle list itself
 (```CYCLES``` can be long / list every cycle on disk, which would make an
 unreadably long directory name) -- the actual cycle list this run
 composited is written to ```validated_cycles.txt``` inside that directory.



The **uuid** is only minted once per distinct set of cycles: before creating a
 new one, every existing ```validation_composite_*``` folder under
 ```MAIN_DIR/CONFIG``` is checked, and if one already has a ```validated_cycles.txt```
 listing exactly this same set of cycles (order doesn't matter), that
 folder is reused -- so re-running this notebook on the same CYCLES keeps
 writing into the same ```COMPOSITE_DIR``` instead of a new random one every
 time. 
 
 **Set SEAFORWARD_COMPOSITE_ID yourself to override this and force a specific id (e.g. to deliberately start a fresh composite for the same cycles)**.
 - replace ```os.environ.get("SEAFORWARD_COMPOSITE_ID")``` with your desired 8-character id, ex: ```"fg03sc40"```

In [4]:

cycle_name = "validated_cycles.txt"

def _find_existing_composite_dir(main_dir, config, cycles):
    wanted = set(cycles)
    base = os.path.join(main_dir, config)
    for d in sorted(glob.glob(os.path.join(base, "validation_composite_*"))):
        f = os.path.join(d, cycle_name)
        if not os.path.isfile(f):
            continue
        with open(f) as fh:
            existing = {line.strip() for line in fh if line.strip()}
        if existing == wanted:
            return d
    return None

FORCE_COMPOSITE_ID = os.environ.get("SEAFORWARD_COMPOSITE_ID")   # <<<< set to force a specific/new id
if FORCE_COMPOSITE_ID:
    COMPOSITE_ID = FORCE_COMPOSITE_ID
    COMPOSITE_DIR = os.path.join(MAIN_DIR, CONFIG, f"validation_composite_{COMPOSITE_ID}")
    print(f"SEAFORWARD_COMPOSITE_ID set - using forced composite directory: {COMPOSITE_DIR}")
else:
    existing_dir = _find_existing_composite_dir(MAIN_DIR, CONFIG, CYCLES)
    if existing_dir:
        COMPOSITE_DIR = existing_dir
        COMPOSITE_ID = os.path.basename(COMPOSITE_DIR).replace("validation_composite_", "")
        print(f"Reusing existing composite directory for this exact cycle set: {COMPOSITE_DIR}")
    else:
        COMPOSITE_ID = uuid.uuid4().hex[:8]
        COMPOSITE_DIR = os.path.join(MAIN_DIR, CONFIG, f"validation_composite_{COMPOSITE_ID}")
        print(f"No existing composite directory matches this cycle set - creating a new one: {COMPOSITE_DIR}")

os.makedirs(COMPOSITE_DIR, exist_ok=True)
with open(os.path.join(COMPOSITE_DIR, cycle_name), "w") as f:
    f.write("\n".join(CYCLES) + "\n")

print(f"Compositing {len(CYCLES)} cycle(s): {CYCLES}")
print(f"Composite outputs -> {COMPOSITE_DIR}")
print(f"  (cycle list also recorded in {os.path.join(COMPOSITE_DIR, cycle_name)})")

Reusing existing composite directory for this exact cycle set: /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6
Compositing 3 cycle(s): ['20260711', '20260723', '20260729']
Composite outputs -> /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6
  (cycle list also recorded in /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6/validated_cycles.txt)


## 1b. Reference-product availability

Same check as 02_validation.ipynb Section 1b -- shared across every
cycle in this composite, since it's a platform-level (not per-cycle)
availability check.


In [5]:
AVAIL = {
    name: cmems.dataset_available(name)
    for name in ("mercator_forecast", "ostia_l4", "odyssea_l3s", "smos_l4_sss")
}

print()
print(AVAIL)


Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:07<00:00,  3.86s/it]


  CMEMS product 'mercator_forecast' (cmems_mod_glo_phy_anfc_0.083deg_P1D-m): available


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:05<00:05,  5.89s/it]

  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available



                                                                                   
Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:08<00:00,  4.10s/it]


  CMEMS product 'odyssea_l3s' (IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE): available


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:06<00:06,  6.46s/it]

  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available

{'mercator_forecast': True, 'ostia_l4': True, 'odyssea_l3s': True, 'smos_l4_sss': True}


## 1c. Resolve paths, download reference data, and label each cycle's
## days by forecast lead time

For each cycle: resolve `CROCO_HIS`/`REFERENCE`, download (or reuse
already-downloaded) the Copernicus Marine Forecast + satellite SST/SSS --
IDENTICAL per-cycle logic to 02_validation.ipynb Section 1c, just looped
over `CYCLES` -- then assign every calendar day in that cycle a
**lead-time label** (`vc.label_cycle_days`, see the intro above).
`cycles_info` (a list of one dict per cycle) is what every composite
function below takes as its first argument.

Numerical-stability check (NaN/Inf in the CROCO output) is done per cycle
here too, matching 02_validation.ipynb Section 1's `stability_ok`.


In [6]:
from datetime import datetime, timedelta

cycles_info = []
stability_ok = True

for cycle in CYCLES:
    print(f"\n== cycle {cycle} ==")
    croco_his, reference, _ = _paths.get_paths(cycle=cycle, config=CONFIG, main_dir=MAIN_DIR)

    ds_c = pp.open_history(croco_his, Yorig=YORIG)
    for v in ("temp", "salt", "zeta"):
        if v in ds_c and (not np.isfinite(ds_c[v].values).all()):
            print(f"  ! NaN/Inf found in {v} - numerical stability check FAILED for this cycle")
            stability_ok = False
    clon_c, clat_c, _ = pp.lonlatmask(ds_c)
    DOMAIN = (float(np.nanmin(clon_c)), float(np.nanmax(clon_c)),
             float(np.nanmin(clat_c)), float(np.nanmax(clat_c)))
    model_times = pd.to_datetime(pp.times(ds_c))
    START_DATE, END_DATE = model_times[0].to_pydatetime(), model_times[-1].to_pydatetime()
    CYCLE_DATE = datetime.strptime(cycle, "%Y%m%d")
    ds_c.close()

    # ---- (i) Copernicus Marine Forecast (Mercator anfc), combined reference file ----
    if not AVAIL['mercator_forecast']:
        print("  Copernicus Marine Forecast unavailable on the CMEMS platform - "
             "this cycle's Sections 2/4/5b will contribute no data.")
    elif cmems.netcdf_covers_time_range(reference, START_DATE, END_DATE, max_step_hours=30):
        print(f"  Mercator: already downloaded and covers the full cycle window: {reference}")
    else:
        if os.path.exists(reference):
            os.remove(reference)
        mercator_dir = os.path.dirname(reference)
        fdays = max((END_DATE.date() - CYCLE_DATE.date()).days, 0)
        cmems.download_mercator_ops(DOMAIN, CYCLE_DATE, hdays=0, fdays=fdays, outputDir=mercator_dir)
        print(f"  Mercator: downloaded -> {reference}" if os.path.exists(reference)
             else f"  Mercator: download ran but {reference} wasn't produced - check {mercator_dir}")

    # ---- (ii) Satellite SST: OSTIA & ODYSSEA, one file per day ----
    sat_files = {}
    for product in ("OSTIA", "ODYSSEA"):
        sat_dir = _paths.satellite_dir(MAIN_DIR, CONFIG, cycle, product)
        sat_files[product] = cmems.download_satellite_sst(product, DOMAIN, START_DATE, END_DATE, sat_dir)

    # ---- (ii-b) Satellite SSS: SMOS L4, one file per day ----
    if not AVAIL['smos_l4_sss']:
        sat_files['SMOS'] = {}
    else:
        smos_dir = _paths.satellite_dir(MAIN_DIR, CONFIG, cycle, "SMOS")
        sat_files['SMOS'] = cmems.download_satellite_sst("SMOS", DOMAIN, START_DATE, END_DATE, smos_dir)
    print()
    
    # ---- lead-time labelling for this cycle's days ----
    days, day_label, label_day = vc.label_cycle_days(croco_his, Yorig=YORIG, spinup_days=SPINUP_DAYS)
    print(f"  days: {days}")
    print(f"  lead labels: {day_label}")

    cycles_info.append({
        "cycle": cycle, "croco_his": croco_his, "reference": reference,
        "days": days, "day_label": day_label, "label_day": label_day,
        "sat_files": sat_files,
    })

LEADS = vc.all_lead_labels(cycles_info)                    # e.g. ['sp1','sp2','fcst1','fcst2','fcst3','fcst4']
FCST_LEADS = vc.all_lead_labels(cycles_info, fcst_only=True)  # spin-up excluded
print(f"\nAll lead labels in this composite : {LEADS}")
print(f"Forecast-only lead labels (Section 5): {FCST_LEADS}")
print(f"Numerical stability across all cycles : {'OK' if stability_ok else 'FAILED - see above'}")
for lead in LEADS:
    print(f"  {lead:6s}: reached by {vc.cycles_reaching(cycles_info, lead)}")


Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:06<00:00,  3.23s/it]


== cycle 20260711 ==


  Mercator: already downloaded and covers the full cycle window: /home/lell/seaforward/forecast/model-runs/Canary_12/20260711/downloaded_data/MERCATOR/MERCATOR_20260711_00.nc


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:06<00:06,  6.86s/it]INFO - 2026-09-05T08:56:41Z - Checking if credentials are valid.


  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available


...

CMEMS: already logged in.
  [ODYSSEA] 2026-07-28: already downloaded - 2026-07-28.nc
  [ODYSSEA] 2026-07-29: already downloaded - 2026-07-29.nc
  [ODYSSEA] 2026-07-30: already downloaded - 2026-07-30.nc
  [ODYSSEA] 2026-07-31: already downloaded - 2026-07-31.nc
  [ODYSSEA] 2026-08-01: already downloaded - 2026-08-01.nc
  [ODYSSEA] 2026-08-02: already downloaded - 2026-08-02.nc
  [ODYSSEA] 2026-08-03: already downloaded - 2026-08-03.nc
  [ODYSSEA] 2026-08-04: already downloaded - 2026-08-04.nc


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:08<00:08,  8.13s/it]INFO - 2026-09-05T08:58:21Z - Checking if credentials are valid.


  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available


INFO - 2026-09-05T08:58:24Z - Valid credentials from configuration file.
Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:10<00:00,  5.22s/it]

CMEMS: already logged in.
  [SMOS] 2026-07-28: already downloaded - 2026-07-28.nc
  [SMOS] 2026-07-29: already downloaded - 2026-07-29.nc
  [SMOS] 2026-07-30: already downloaded - 2026-07-30.nc
  [SMOS] 2026-07-31: already downloaded - 2026-07-31.nc
  [SMOS] 2026-08-01: already downloaded - 2026-08-01.nc
  [SMOS] 2026-08-02: already downloaded - 2026-08-02.nc
  [SMOS] 2026-08-03: already downloaded - 2026-08-03.nc
  [SMOS] 2026-08-04: already downloaded - 2026-08-04.nc

  days: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']
  lead labels: {'2026-07-28': 'sp1', '2026-07-29': 'sp2', '2026-07-30': 'fcst1', '2026-07-31': 'fcst2', '2026-08-01': 'fcst3', '2026-08-02': 'fcst4', '2026-08-03': 'fcst5', '2026-08-04': 'fcst6'}

All lead labels in this composite : ['sp1', 'sp2', 'fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']
Forecast-only lead labels (Section 5): ['fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']
Numerica

## 2. Composite bias maps -- CROCO forecast vs Copernicus Marine
## Forecast, by lead time

Composite counterpart to 02_validation.ipynb Section 2: instead of one
2x2 figure per CALENDAR DAY of a single cycle, ONE 2x2 figure per LEAD
TIME (`sp1`, `sp2`, `fcst1`, `fcst2`, ...), composited (pixel-averaged)
across every cycle in `cycles_info` that reaches that lead. A given
lead's figure shows the composite-mean CROCO field, composite-mean
Copernicus field, the mean bias across contributing cycles, and the
RMSE across contributing cycles (same accumulation math as
`sftools.validation._multiday_bias_rmse`, just one sample per
CONTRIBUTING CYCLE instead of one per day within a single cycle).

`sst_stats_by_lead`/`ssh_stats_by_lead`/`salt_stats_by_lead` mirror
02_validation.ipynb's `*_stats_by_day` dicts, keyed by lead instead of date.


In [7]:
if AVAIL['mercator_forecast']:
    sst_stats_by_lead = {}
    for lead in LEADS:
        res = vc.compare_composite_field(
            cycles_info, var='temp', lead=lead, Yorig=YORIG, depth_m=DEPTH_M,daily_mean=True,
            out=os.path.join(COMPOSITE_DIR, f'sst_vs_forecast_{lead}.png'))
        if res is not None:
            _, sst_stats_by_lead[lead] = res
    print(f"-> {len(sst_stats_by_lead)} figure(s) written, one per lead: {list(sst_stats_by_lead)}")
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")


temperature @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.116  RMSE=0.225  cRMSE=0.192  corr=0.997
temperature @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.194  RMSE=0.320  cRMSE=0.255  corr=0.995
temperature @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.197  RMSE=0.344  cRMSE=0.282  corr=0.993
temperature @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.352  RMSE=0.470  cRMSE=0.311  corr=0.990
temperature @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.431  RMSE=0.536  cRMSE=0.318  corr=0.988
temperature @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.541  RMSE=0.635  cRMSE=0.332  corr=0.986
temperature @ fcst5  (composite of 1 cycle(s

In [8]:
if AVAIL['mercator_forecast']:
    # SSH has no depth dimension -- always surface, regardless of DEPTH_M.
    ssh_stats_by_lead = {}
    for lead in LEADS:
        res = vc.compare_composite_field(
            cycles_info, var='ssh', lead=lead, Yorig=YORIG,daily_mean=True,
            out=os.path.join(COMPOSITE_DIR, f'ssh_vs_forecast_{lead}.png'))
        if res is not None:
            _, ssh_stats_by_lead[lead] = res
    print(f"-> {len(ssh_stats_by_lead)} figure(s) written, one per lead: {list(ssh_stats_by_lead)}")
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")


SSH anomaly @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.006  cRMSE=0.006  corr=0.995
SSH anomaly @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.007  cRMSE=0.007  corr=0.992
SSH anomaly @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.008  cRMSE=0.008  corr=0.990
SSH anomaly @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.009  cRMSE=0.009  corr=0.987
SSH anomaly @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.011  cRMSE=0.011  corr=0.982
SSH anomaly @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.013  cRMSE=0.013  corr=0.976
SSH anomaly @ fcst5  (composite of 1 cycle(s

In [9]:
if AVAIL['mercator_forecast']:
    salt_stats_by_lead = {}
    for lead in LEADS:
        res = vc.compare_composite_field(
            cycles_info, var='salt', lead=lead, Yorig=YORIG, depth_m=DEPTH_M,daily_mean=True,
            out=os.path.join(COMPOSITE_DIR, f'sss_vs_forecast_{lead}.png'))
        if res is not None:
            _, salt_stats_by_lead[lead] = res
    print(f"-> {len(salt_stats_by_lead)} figure(s) written, one per lead: {list(salt_stats_by_lead)}")
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")


salinity @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.013  RMSE=0.059  cRMSE=0.058  corr=0.989
salinity @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.017  RMSE=0.076  cRMSE=0.074  corr=0.981
salinity @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.022  RMSE=0.092  cRMSE=0.089  corr=0.971
salinity @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.030  RMSE=0.115  cRMSE=0.111  corr=0.960
salinity @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.033  RMSE=0.130  cRMSE=0.126  corr=0.948
salinity @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.036  RMSE=0.140  cRMSE=0.136  corr=0.940
salinity @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [salinity]  n=8209 

In [10]:
if AVAIL['mercator_forecast']:
    cur_stats_by_lead = {}
    for lead in LEADS:
        res = vc.compare_composite_field(
            cycles_info, var='speed', lead=lead, Yorig=YORIG, depth_m=DEPTH_M,daily_mean=True,
            out=os.path.join(COMPOSITE_DIR, f'currents_vs_forecast_{lead}.png'))
        if res is not None:
            _, cur_stats_by_lead[lead] = res
    print(f"-> {len(cur_stats_by_lead)} figure(s) written, one per lead: {list(cur_stats_by_lead)}")
    print()
    print("Surface velocities pass criterion is QUALITATIVE (visual consistency with")
    print("expected gyre/coastal-jet circulation) -- inspect the vector maps")
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")


speed @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.008  RMSE=0.047  cRMSE=0.046  corr=0.828
speed @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.010  RMSE=0.039  cRMSE=0.037  corr=0.859
speed @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.024  RMSE=0.050  cRMSE=0.044  corr=0.850
speed @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.001  RMSE=0.048  cRMSE=0.048  corr=0.792
speed @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=-0.015  RMSE=0.053  cRMSE=0.051  corr=0.764
speed @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=-0.014  RMSE=0.063  cRMSE=0.061  corr=0.603
speed @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [speed]  n=8209  bias=-0.007  RMSE=0.091  cRMSE=0.091  cor

## 3. Composite scatter plots -- pointwise CROCO vs Copernicus Marine
## Forecast, by lead time

Composite counterpart to 02_validation.ipynb Section 3: one SST+SSH
scatter figure per lead time, with points POOLED (concatenated) from
every cycle that reaches that lead -- so, e.g., `fcst2`'s scatter
combines every contributing cycle's 3rd-day points into one distribution
and one bias/RMSE/corr annotation, rather than scoring each cycle
separately.


In [11]:
if AVAIL['mercator_forecast']:
    scatter_figs = {}
    for lead in LEADS:
        out_png = os.path.join(COMPOSITE_DIR, f"scatter_sst_ssh_vs_forecast_{lead}.png")
        res = vc.scatter_composite(cycles_info, lead, Yorig=YORIG, daily_mean=True, out=out_png)
        if res is not None:
            scatter_figs[lead] = out_png
    print(f"-> {len(scatter_figs)} figure(s) written, one per lead: {list(scatter_figs)}")
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")


-> 8 figure(s) written, one per lead: ['sp1', 'sp2', 'fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']


## 4. Composite GODAE scorecard & Taylor diagram, by lead time

Composite counterpart to 02_validation.ipynb Section 4: for each lead
time, the GODAE scorecard (`bias`/`RMSD`/`uRMSD`/`corr`/`SI`/`std_ratio`)
for every variable (`temp`, `ssh`, `salt`, `speed`) is computed on points
POOLED across every contributing cycle (`vc.godae_scorecard_composite`),
then one Taylor diagram is drawn per lead, summarising all four
variables' pooled skill at that lead time.


In [12]:
if AVAIL['mercator_forecast']:
    rows = []
    report_by_lead = {}
    for lead in LEADS:
        lead_rows = []
        for var in ('temp', 'ssh', 'salt', 'speed'):
            s = vc.godae_scorecard_composite(cycles_info, var, lead, depth_m=DEPTH_M, Yorig=YORIG)
            if s is not None:
                lead_rows.append({**s, 'vs': 'reference', 'layer': 'all'})
        if lead_rows:
            report_by_lead[lead] = pd.DataFrame(lead_rows)
            rows.extend(lead_rows)
            print(f"-- {lead}  (cycles: {vc.cycles_reaching(cycles_info, lead)}) --")
            val.print_scorecard_table(report_by_lead[lead])

    # every lead x every variable; Section 5's pass/fail summary below uses
    # only the fcst* leads (FCST_LEADS), spin-up (sp1/sp2) excluded.
    report = pd.DataFrame(rows)
else:
    print("Copernicus Marine Forecast unavailable - skipping GODAE scorecard "
         "(Sections 4/5 below will be skipped too).")
    report = pd.DataFrame(columns=['variable', 'lead', 'bias', 'rmsd', 'urmsd', 'corr', 'std_ratio', 'vs', 'layer'])
    report_by_lead = {}


-- sp1  (cycles: ['20260711', '20260723', '20260729']) --
variable        vs layer     n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 24627  0.013 0.291  0.291 0.992  1.158  12.955
     ssh reference   all 24627 -0.000 0.009  0.009 0.985 23.268  17.204
    salt reference   all 24627  0.016 0.082  0.081 0.978  0.224  21.428
   speed reference   all 24627  0.045 0.110  0.101 0.535 59.838 100.270
-- sp2  (cycles: ['20260711', '20260723', '20260729']) --
variable        vs layer     n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 24627 -0.046 0.331  0.327 0.990  1.300  14.585
     ssh reference   all 24627 -0.000 0.010  0.010 0.983 24.685  18.355
    salt reference   all 24627  0.020 0.098  0.096 0.968  0.265  25.683
   speed reference   all 24627  0.058 0.110  0.094 0.624 53.650 100.284
-- fcst1  (cycles: ['20260711', '20260723', '20260729']) --
variable        vs layer     n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 24627 -

In [13]:
if not report.empty:
    taylor_figs = {}
    for lead, rep in report_by_lead.items():
        fig = plt.figure(figsize=(7, 7))
        ax = fig.add_subplot(111, polar=True)
        has_neg = bool((rep['corr'] < 0).any())
        thetamax = 180 if has_neg else 90
        corr_ticks = ([-1.0, -0.5, 0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0] if has_neg
                      else [0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0])
        ax.set_thetamin(0); ax.set_thetamax(thetamax)
        ax.set_xticks(np.arccos(corr_ticks)); ax.set_xticklabels([str(c) for c in corr_ticks])
        ax.set_rlabel_position(0)   # keep the std-dev tick labels along the bottom (theta=0) axis
        finite = rep['std_ratio'][np.isfinite(rep['std_ratio'])]
        r_max = max(1.6, finite.max() * 1.2) if len(finite) else 1.6
        ax.set_ylim(0, r_max)
        ax.plot(0, 1, 'k*', ms=16, label='reference')
        for _, row in rep.iterrows():
            theta = np.arccos(np.clip(row['corr'], -1, 1))
            ax.plot(theta, row['std_ratio'], 'o', ms=10,
                   label=f"{row['variable']}  (n_cycles={row['n_cycles']}, RMSD={row['rmsd']:.2f})")

        ax.text(0.5, -0.08, 'Normalised standard deviation (CROCO / reference)',
               transform=ax.transAxes, ha='center', va='top', fontsize=10)
        ax.text(np.radians(thetamax / 6), r_max * 1.25, 'Correlation coefficient',
               ha='center', va='center', fontsize=10,
               rotation=90 - thetamax / 2, rotation_mode='anchor')

        ax.set_title(f'Taylor diagram -- CROCO vs reference ({lead}, composite)', pad=20)
        ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=9)
        fig.tight_layout()
        out_png = os.path.join(COMPOSITE_DIR, f"taylor_diagram_{lead}.png")
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        taylor_figs[lead] = out_png
    print(f"-> {len(taylor_figs)} figure(s) written, one per lead: {list(taylor_figs)}")
else:
    print("No GODAE scorecard available (Copernicus Marine Forecast unavailable) - skipping Taylor diagram.")


-> 8 figure(s) written, one per lead: ['sp1', 'sp2', 'fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']


## 5. Automated pass/fail summary (V1) 
**-- composite, forecast leads only**

Composite counterpart to 02_validation.ipynb Section 5, with one
simplification: since spin-up is already a SEPARATE set of lead labels
(`sp1`, `sp2`, ...) rather than the first N calendar days of one cycle,
excluding it is just `report[report['lead'].isin(FCST_LEADS)]` -- no
day-counting/slicing needed. Each criterion is scored against the mean
across every (cycle, fcst-lead) pair pooled in `FCST_LEADS`.


In [14]:
if not report.empty:
    report_eval = report[report['lead'].isin(FCST_LEADS)]

    by_var = (report_eval.groupby('variable')[['bias', 'rmsd', 'urmsd', 'corr', 'si', 'si_std', 'std_ratio']]
              .mean().to_dict('index'))

    criteria = [
        ('SST composite-mean domain-avg RMSD < 0.5 degC',       by_var['temp']['rmsd'] < 0.5,  f"{by_var['temp']['rmsd']:.3f} degC"),
        ('SSH composite-mean spatial correlation > 0.90',       by_var['ssh']['corr'] > 0.90, f"{by_var['ssh']['corr']:.3f}"),
        ('Salinity composite-mean domain-avg |bias| < 0.2 PSU', abs(by_var['salt']['bias']) < 0.2, f"{by_var['salt']['bias']:+.3f} PSU"),
        ('Numerical stability (no NaN/Inf, every cycle)',       stability_ok, 'see section 1c'),
    ]

    print(f"Composite scorecard: {len(CYCLES)} cycle(s) {CYCLES}, forecast leads only "
         f"(spin-up {[f'sp{i+1}' for i in range(SPINUP_DAYS)]} excluded): {FCST_LEADS}")
    print()
    print(f"{'Criterion':50s} {'Result':6s}  Value")
    print('-' * 75)
    all_pass = True
    for name, passed, value in criteria:
        all_pass &= passed
        print(f"{name:50s} {'PASS' if passed else 'FAIL':6s}  {value}")
    print('-' * 75)
    print()

    # Worst single LEAD (among fcst leads) per headline variable -- a
    # composite mean can mask one bad lead time; this flags it, and
    # Section 5b's boxplot shows the full spread.
    if len(FCST_LEADS) > 1:
        temp_rows = report_eval[report_eval['variable'] == 'temp']
        ssh_rows  = report_eval[report_eval['variable'] == 'ssh']
        salt_rows = report_eval[report_eval['variable'] == 'salt']
        worst_temp = temp_rows.loc[temp_rows['rmsd'].idxmax()]
        worst_ssh  = ssh_rows.loc[ssh_rows['corr'].idxmin()]
        worst_salt = salt_rows.loc[salt_rows['bias'].abs().idxmax()]
        print()
        print(f"Worst single lead -- SST RMSD:   {worst_temp['lead']}  ({worst_temp['rmsd']:.3f} degC)")
        print(f"Worst single lead -- SSH corr:   {worst_ssh['lead']}  ({worst_ssh['corr']:.3f})")
        print(f"Worst single lead -- Salt bias:  {worst_salt['lead']}  ({worst_salt['bias']:+.3f} PSU)")

    print()
    print("Note: surface-current skill (by_var['speed']) has no fixed Section 9.3")
    print("threshold -- its criterion is qualitative (inspect the vector maps in")
    print("Section 2), same as 02_validation.ipynb.")
else:
    print("Overall V1 status: SKIPPED -- Copernicus Marine Forecast unavailable, "
         "no GODAE scorecard to build the pass/fail summary from.")


Composite scorecard: 3 cycle(s) ['20260711', '20260723', '20260729'], forecast leads only (spin-up ['sp1', 'sp2'] excluded): ['fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']

Criterion                                          Result  Value
---------------------------------------------------------------------------
SST composite-mean domain-avg RMSD < 0.5 degC      FAIL    0.569 degC
SSH composite-mean spatial correlation > 0.90      PASS    0.953
Salinity composite-mean domain-avg |bias| < 0.2 PSU PASS    +0.036 PSU
Numerical stability (no NaN/Inf, every cycle)      PASS    see section 1c
---------------------------------------------------------------------------


Worst single lead -- SST RMSD:   fcst6  (0.864 degC)
Worst single lead -- SSH corr:   fcst6  (0.914)
Worst single lead -- Salt bias:  fcst5  (+0.045 PSU)

Note: surface-current skill (by_var['speed']) has no fixed Section 9.3
threshold -- its criterion is qualitative (inspect the vector maps in
Section 2), same as 02_v

### 5b. Composite domain-wide bias boxplot, by lead time

Composite counterpart to 02_validation.ipynb Section 5b: one box per
LEAD TIME (spin-up included, so the transient is visible), pooling
`vc.composite_domain_diff` across every contributing cycle at that lead --
plotted with the exact same `sftools.validation.bias_boxplot` used by the
single-cycle notebook (no separate composite plotting code needed).


In [15]:
if AVAIL['mercator_forecast']:
    dlab = 'surface' if DEPTH_M is None else f'{DEPTH_M:g} m'

    ssh_diffs   = [vc.composite_domain_diff(cycles_info, 'ssh',   lead, Yorig=YORIG, daily_mean=True) for lead in LEADS]
    temp_diffs  = [vc.composite_domain_diff(cycles_info, 'temp',  lead, depth_m=DEPTH_M, Yorig=YORIG, daily_mean=True) for lead in LEADS]
    salt_diffs  = [vc.composite_domain_diff(cycles_info, 'salt',  lead, depth_m=DEPTH_M, Yorig=YORIG, daily_mean=True) for lead in LEADS]
    speed_diffs = [vc.composite_domain_diff(cycles_info, 'speed', lead, depth_m=DEPTH_M, Yorig=YORIG, daily_mean=True) for lead in LEADS]

    def _stacked_boxplot(diffs_list, ylabels, titles, out, metric='bias'):
        data = [[np.abs(d) for d in diffs] for diffs in diffs_list] if metric == 'rmse' else diffs_list
        fig, axes = plt.subplots(len(data), 1, figsize=(max(6, 1.1 * len(LEADS)), 3.3 * len(data)),
                                 sharex=True)
        positions = np.arange(1, len(LEADS) + 1)
        for ax, diffs, ylabel, title in zip(axes, data, ylabels, titles):
            ax.boxplot(diffs, positions=positions, showfliers=False)
            ax.set_xticks(positions); ax.set_xticklabels(LEADS)
            if metric == 'bias':
                ax.axhline(0, color='k', ls='--', lw=1)
            ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(alpha=0.3)
        axes[-1].set_xlabel('lead time')
        plt.setp(axes[-1].get_xticklabels(), rotation=45, ha='right')
        fig.suptitle(f"CROCO - parent  (composite of {len(CYCLES)} cycles)" if metric == 'bias'
                    else f"|CROCO - parent|  (composite of {len(CYCLES)} cycles)")
        fig.tight_layout()
        fig.savefig(out, dpi=150, bbox_inches='tight')
        return fig

    diffs_all = [ssh_diffs, temp_diffs, salt_diffs, speed_diffs]
    ylabels = ["SSH' bias (m)", 'temperature bias (degC)', 'salinity bias (PSU)', 'speed bias (m s$^{-1}$)']
    titles = ['SSH anomaly bias', f'temperature bias  ({dlab})', f'salinity bias  ({dlab})',
             f'current speed bias  ({dlab})']

    _stacked_boxplot(diffs_all, ylabels, titles,
                     os.path.join(COMPOSITE_DIR, 'boxplot_bias_vs_forecast_composite.png'), metric='bias')

    rmse_ylabels = [y.replace('bias', '|error|') for y in ylabels]
    rmse_titles = [t.replace('bias', 'RMSE-spread') for t in titles]
    _stacked_boxplot(diffs_all, rmse_ylabels, rmse_titles,
                     os.path.join(COMPOSITE_DIR, 'boxplot_rmse_vs_forecast_composite.png'), metric='rmse')
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")


### 5c. Composite CROCO-vs-satellite domain-wide bias boxplot, by lead time

Composite counterpart to 02_validation.ipynb Section 5c/6c: OSTIA and
ODYSSEA grouped side by side, SMOS on its own boxplot, one box-group per
LEAD TIME, pooling `vc.composite_domain_diff_satellite` across every
contributing cycle -- plotted with `sftools.validation.bias_boxplot_multi`,
same as the single-cycle notebook.


In [16]:
sat_avail_key = {"OSTIA": "ostia_l4", "ODYSSEA": "odyssea_l3s", "SMOS": "smos_l4_sss"}
any_sat_avail = any(AVAIL.get(sat_avail_key[p], False) and any(c["sat_files"].get(p) for c in cycles_info)
                    for p in sat_avail_key)

if not any_sat_avail:
    print("No satellite product (OSTIA/ODYSSEA/SMOS) available/downloaded for this composite - skipping.")
else:
    ostia_diffs   = [vc.composite_domain_diff_satellite(cycles_info, 'OSTIA',   lead, Yorig=YORIG) for lead in LEADS]
    odyssea_diffs = [vc.composite_domain_diff_satellite(cycles_info, 'ODYSSEA', lead, Yorig=YORIG) for lead in LEADS]
    smos_diffs    = [vc.composite_domain_diff_satellite(cycles_info, 'SMOS',    lead, Yorig=YORIG) for lead in LEADS]

    sst_groups = {"OSTIA": ostia_diffs, "ODYSSEA": odyssea_diffs}
    sss_groups = {"SMOS": smos_diffs}

    val.bias_boxplot_multi(sst_groups, LEADS, 'temperature bias (degC)',
                           f'CROCO - satellite SST bias  (composite of {len(CYCLES)} cycles)',
                           colors=['C1', 'C2'],
                           out=os.path.join(COMPOSITE_DIR, 'boxplot_bias_sst_vs_satellite_composite.png'))
    val.bias_boxplot_multi(sss_groups, LEADS, 'salinity bias (PSU)',
                           f'CROCO - satellite SSS bias  (composite of {len(CYCLES)} cycles)',
                           colors=['C4'],
                           out=os.path.join(COMPOSITE_DIR, 'boxplot_bias_sss_vs_satellite_composite.png'))

    val.bias_boxplot_multi(sst_groups, LEADS, 'temperature |error| (degC)',
                           f'CROCO - satellite SST RMSE-spread  (composite of {len(CYCLES)} cycles)',
                           colors=['C1', 'C2'], metric='rmse',
                           out=os.path.join(COMPOSITE_DIR, 'boxplot_rmse_sst_vs_satellite_composite.png'))
    val.bias_boxplot_multi(sss_groups, LEADS, 'salinity |error| (PSU)',
                           f'CROCO - satellite SSS RMSE-spread  (composite of {len(CYCLES)} cycles)',
                           colors=['C4'], metric='rmse',
                           out=os.path.join(COMPOSITE_DIR, 'boxplot_rmse_sss_vs_satellite_composite.png'))


### 5d. Composite point-wide mean timeseries, by lead time

Composite mean timeseries (CROCO vs parent, CROCO vs satellite)

Point + full-domain mean +/- 1 std (fill_between) per lead, for CROCO vs
Copernicus Marine Forecast and CROCO vs satellite -- set `POINT_LON`/
`POINT_LAT` below.

In [17]:
if AVAIL['mercator_forecast']:
    vars_specs = [('ssh', "SSH' (m)", 'SSH anomaly', None),
                 ('temp', 'temperature (degC)', f'temperature ({dlab})', DEPTH_M),
                 ('salt', 'salinity (PSU)', f'salinity ({dlab})', DEPTH_M),
                 ('speed', 'speed (m s$^{-1}$)', f'current speed ({dlab})', DEPTH_M)]

    domain_rows = []
    point_rows = []
    for var, ylabel, title, dm in vars_specs:
        cd, pd_ = vc.composite_domain_series(cycles_info, var, LEADS, depth_m=dm, Yorig=YORIG, daily_mean=True)
        domain_rows.append((ylabel, title + ' (full domain)', cd, pd_))
        cp, pp_ = vc.composite_point_series(cycles_info, var, POINT_LON, POINT_LAT, LEADS, depth_m=dm, Yorig=YORIG, daily_mean=True)
        point_rows.append((ylabel, title + f' ({POINT_LON:.2f}, {POINT_LAT:.2f})', cp, pp_))

    ## full-domain averages over every horizontal level, cumulating localized errors. 
    ## The figure shape can then show huge std spread around the mean.
    vc.composite_two_series_timeseries(domain_rows, LEADS, len(CYCLES), labels=("CROCO", "parent"),
                                       out=os.path.join(COMPOSITE_DIR, "timeseries_mean_domain_vs_forecast_composite.png"))
    ## single point
    vc.composite_two_series_timeseries(point_rows, LEADS, len(CYCLES), labels=("CROCO", "parent"),
                                       out=os.path.join(COMPOSITE_DIR, "timeseries_mean_point_vs_forecast_composite.png"))
else:
    print("Copernicus Marine Forecast unavailable - skipping this comparison.")

if not any_sat_avail:
    print("No satellite product available/downloaded for this composite - skipping.")
else:
    sat_domain_rows, sat_point_rows = [], []
    for product, ylabel in (('OSTIA', 'SST bias (degC)'), ('ODYSSEA', 'SST bias (degC)'), ('SMOS', 'SSS (PSU)')):
        cd, sd = vc.composite_satellite_series(cycles_info, product, LEADS, Yorig=YORIG)
        sat_domain_rows.append((ylabel, f'CROCO vs {product} (full domain)', cd, sd))
        cp, sp = vc.composite_satellite_series(cycles_info, product, LEADS, lon0=POINT_LON, lat0=POINT_LAT, Yorig=YORIG)
        sat_point_rows.append((ylabel, f'CROCO vs {product} ({POINT_LON:.2f}, {POINT_LAT:.2f})', cp, sp))
    
    ## full-domain averages over every horizontal level, cumulating localized errors. 
    ## The figure shape can then show huge std spread around the mean.
    vc.composite_two_series_timeseries(sat_domain_rows, LEADS, len(CYCLES), labels=("CROCO", "satellite"),
                                       out=os.path.join(COMPOSITE_DIR, "timeseries_mean_domain_vs_satellite_composite.png"))
    ## single point
    vc.composite_two_series_timeseries(sat_point_rows, LEADS, len(CYCLES), labels=("CROCO", "satellite"),
                                       out=os.path.join(COMPOSITE_DIR, "timeseries_mean_point_vs_satellite_composite.png"))
    

### 5e. Composite vertical profiles (point and full-domain), temp/salt/speed

Mean +/- 1 std (fill_between) across every cycle reaching `PROFILE_LEAD`,
CROCO vs parent.

In [18]:
for PROFILE_LEAD in LEADS:
    for var in ('temp', 'salt', 'speed'):
        vc.composite_point_profile(cycles_info, var, POINT_LON, POINT_LAT, PROFILE_LEAD,
                                   Yorig=YORIG, daily_mean=True,
                                   out=os.path.join(COMPOSITE_DIR, f'profile_point_{var}_{PROFILE_LEAD}_composite.png'))

        # you can also draw full-domain vertical profile. But this will show up important difference 
        # between CROCO and reference. A full-domain mean vertical profile averages over every horizontal point at each depth, 
        # cumulating localized errors at every level. The profile shape then shows clear difference betwwen CROCO and reference
        vc.composite_domain_profile(cycles_info, var, PROFILE_LEAD, Yorig=YORIG, daily_mean=True,
                                    out=os.path.join(COMPOSITE_DIR, f'profile_domain_{var}_{PROFILE_LEAD}_composite.png'))


## 6. Composite satellite SST validation map -- OSTIA & ODYSSEA, by lead time

Composite counterpart to 02_validation.ipynb Section 7: one 2x2 figure
per lead time (CROCO composite mean | satellite composite mean / bias |
RMSE), composited across every cycle with a downloaded file for that
lead's day.


In [19]:
sat_stats_all = {}
for product in ("OSTIA", "ODYSSEA"):
    if not AVAIL[sat_avail_key[product]]:
        print(f"{product} unavailable on the CMEMS platform - skipping.")
        continue
    print(f"\n-- {product} --")
    figs, rows = {}, []
    for lead in LEADS:
        out_png = os.path.join(COMPOSITE_DIR, f"sst_vs_{product.lower()}_{lead}.png")
        res, s = vc.compare_composite_satellite_grid(cycles_info, product, lead, Yorig=YORIG, out=out_png)
        if res is not None:
            figs[lead] = res
            rows.append(s)
    stats = pd.DataFrame(rows)
    sat_stats_all[product] = stats
    print(f"  -> {len(figs)} figure(s) written, one per lead: {list(figs)}")
    if not stats.empty:
        stats.to_csv(os.path.join(COMPOSITE_DIR, f"sst_vs_{product.lower()}_stats_composite.csv"), index=False)
print()



-- OSTIA --
[OSTIA] SST @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=+0.186  RMSE=0.446  cRMSE=0.405  corr=0.981
[OSTIA] SST @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=+0.163  RMSE=0.446  cRMSE=0.415  corr=0.980
[OSTIA] SST @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=+0.191  RMSE=0.465  cRMSE=0.424  corr=0.978
[OSTIA] SST @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=-0.053  RMSE=0.465  cRMSE=0.462  corr=0.976
[OSTIA] SST @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=-0.063  RMSE=0.444  cRMSE=0.439  corr=0.978
[OSTIA] SST @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=-0.089  RMSE=0.499  cRMSE=0.491  corr=0.973
[OSTIA] SST @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [SST]  n=8257  

## 6b. Composite satellite SSS validation -- SMOS, by lead time

Composite counterpart to 02_validation.ipynb Section 7b -- same pattern
as Section 6 above, for SMOS SSS instead of OSTIA/ODYSSEA SST.


In [20]:
if not AVAIL['smos_l4_sss']:
    print("SMOS SSS unavailable on the CMEMS platform - skipping.")
elif not any(c["sat_files"].get("SMOS") for c in cycles_info):
    print("SMOS: nothing downloaded for any cycle in this composite - skipping.")
else:
    figs, rows = {}, []
    for lead in LEADS:
        out_png = os.path.join(COMPOSITE_DIR, f"sss_vs_smos_{lead}.png")
        res, s = vc.compare_composite_satellite_grid(cycles_info, "SMOS", lead, Yorig=YORIG, out=out_png)
        if res is not None:
            figs[lead] = res
            rows.append(s)
    smos_stats = pd.DataFrame(rows)
    sat_stats_all["SMOS"] = smos_stats
    print(f"  -> {len(figs)} figure(s) written, one per lead: {list(figs)}")
    if not smos_stats.empty:
        smos_stats.to_csv(os.path.join(COMPOSITE_DIR, "sss_vs_smos_stats_composite.csv"), index=False)
print()


[SMOS] SSS @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.091  RMSE=0.263  cRMSE=0.247  corr=0.664
[SMOS] SSS @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.085  RMSE=0.259  cRMSE=0.245  corr=0.638
[SMOS] SSS @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.077  RMSE=0.272  cRMSE=0.261  corr=0.562
[SMOS] SSS @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.082  RMSE=0.290  cRMSE=0.278  corr=0.463
[SMOS] SSS @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.066  RMSE=0.265  cRMSE=0.257  corr=0.542
[SMOS] SSS @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.074  RMSE=0.232  cRMSE=0.220  corr=0.674
[SMOS] SSS @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [SSS]  n=8070  bias=-0.070  RMSE=0.

## 7. Composite HTML summary

Composite counterpart to 02_validation.ipynb Section 9: gathers every
figure/table this notebook wrote into `COMPOSITE_DIR` into one
self-contained, offline-viewable HTML page, via
`vc.build_html_summary_composite` -- a thin wrapper around the SAME
`val.build_html_summary` the single-cycle notebook uses, so the report
layout/lightbox/grouping code is never duplicated.

In [21]:
html_path = vc.build_html_summary_composite(
    COMPOSITE_DIR, composite_id=COMPOSITE_ID, config=CONFIG,
    cycles=CYCLES,)
print(f"Open {html_path} in a browser to review this composite.")

Open /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6/validation_cycle_composite_bce243d6.html in a browser to review this composite.


---
## Notes

- **Merging rule**: everything in this notebook merges cycles by **lead
  label**, not calendar date -- `sp1`/`sp2` = model spin-up (first
  `SPINUP_DAYS` days of every cycle), `fcst1`, `fcst2`, ... = forecast
  lead day 1, 2, ... A cycle that doesn't reach a given lead (shorter
  forecast) is simply skipped for that lead, not raised as an error, so
  cycles of different length can be composited together.
  
- **In-situ (Section 8 of 02_validation.ipynb)** are not yet ported to the composite layer -- both
  are natural extensions (in-situ collocation is already
  per-observation/per-platform rather than per-grid-point, so pooling by
  lead time would need the platform's own timestamp reinterpreted as a
  lead relative to its cycle's `CYCLE_DATE`.
